In [1]:
%cd ..
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

sys.path.insert(0, Path().absolute().parent.as_posix())

/Users/ruizhechao/Documents/NNforHJB


In [2]:
import numpy as np
import torch
import random

In [3]:
path = 'rawdata/raw_data/data/VDP_beta_0.1_grid_30x30.npy'
data = np.load(path)
len(data)

900

In [4]:
data_dict = {
    "x": np.asarray(data["x"], dtype=np.float64),
    "v": np.asarray(data["v"], dtype=np.float64),
    "dv": np.asarray(data["dv"], dtype=np.float64),
}

## PDPA v3: Finite-step insertion for |c|^q penalty

Gamma sweep with `power=3.0` (`q = 2/(p+1) = 0.5`). Uses the finite-step insertion criterion `Delta J(c*; omega) < 0` instead of `|p(omega)| > alpha`.

### Hyperparameters

In [ ]:
seed = 42

gammas = [0, 1e-2, 1e-1, 1, 10]
alpha = 1e-5
power = 3.0
activation = torch.relu
use_sphere = True

num_iterations = 10
num_insertions = 50
pruning_threshold = 1e-5

In [6]:
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

#### L2 loss

In [7]:
from src.PDPA_v3 import PDPA_v3

results_l2 = []

for gamma in gammas:
    pdpa = PDPA_v3(
        data=data_dict,
        alpha=alpha,
        gamma=gamma,
        power=power,
        activation=activation,
        use_sphere=use_sphere,
        loss_weights="l2",
        verbose=True,
    )

    result = pdpa.retrain(
        num_iterations=num_iterations,
        num_insertion=num_insertions,
        threshold=pruning_threshold,
        verbose=True,
    )
    results_l2.append(result)

2026-04-13 12:12:02.480 | INFO     | src.model:__init__:72 - Model initialized
2026-04-13 12:12:02.481 | INFO     | src.model:_prepare_data:107 - Training set: 899 samples, Validation set: 1 samples
2026-04-13 12:12:02.482 | INFO     | src.PDPA_v3:retrain:597 - Initialization
2026-04-13 12:12:03.817 | INFO     | src.PDPA_v3:insertion:397 - insertion: tried 50, after merge 2, accepted 2 (capped <=15, alpha=1e-05, q=0.500)
2026-04-13 12:12:03.818 | INFO     | src.model:_setup_optimizer:186 - Using SSN optimizer with alpha=1e-05, gamma=0, th=0.5, lr =1.0
2026-04-13 12:12:03.818 | INFO     | src.model:train:299 - Starting network training session
2026-04-13 12:12:03.824 | INFO     | src.model:train:385 - Epoch 0: Train Loss = 1.039647, Val Loss = 4.781287
2026-04-13 12:12:03.827 | INFO     | src.model:train:385 - Epoch 2: Train Loss = 1.039647, Val Loss = 4.781287
2026-04-13 12:12:03.829 | INFO     | src.model:train:385 - Epoch 4: Train Loss = 1.039647, Val Loss = 4.781287
2026-04-13 12:12

In [8]:
import pickle

with open("models/experiment_11.pkl", "wb") as f:
    pickle.dump(results_l2, f)

print([r["best_iteration"] for r in results_l2])
print([r["final_neurons"] for r in results_l2])

[9, 9, 9, 9, 9]
[29, 43, 37, 33, 29]


#### H1 loss

In [9]:
from src.PDPA_v3 import PDPA_v3

results_h1 = []

for gamma in gammas:
    pdpa = PDPA_v3(
        data=data_dict,
        alpha=alpha,
        gamma=gamma,
        power=power,
        activation=activation,
        use_sphere=use_sphere,
        loss_weights="h1",
        verbose=True,
    )

    result = pdpa.retrain(
        num_iterations=num_iterations,
        num_insertion=num_insertions,
        threshold=pruning_threshold,
        decorrelation=False,
        verbose=True,
    )
    results_h1.append(result)

2026-04-13 12:12:52.007 | INFO     | src.model:__init__:72 - Model initialized
2026-04-13 12:12:52.008 | INFO     | src.model:_prepare_data:107 - Training set: 899 samples, Validation set: 1 samples
2026-04-13 12:12:52.010 | INFO     | src.PDPA_v3:retrain:597 - Initialization
2026-04-13 12:12:52.969 | INFO     | src.PDPA_v3:insertion:397 - insertion: tried 50, after merge 3, accepted 2 (capped <=15, alpha=1e-05, q=0.500)
2026-04-13 12:12:52.970 | INFO     | src.model:_setup_optimizer:186 - Using SSN optimizer with alpha=1e-05, gamma=0, th=0.5, lr =1.0
2026-04-13 12:12:52.970 | INFO     | src.model:train:299 - Starting network training session
2026-04-13 12:12:52.972 | INFO     | src.model:train:385 - Epoch 0: Train Loss = 2.033226, Val Loss = 14.389959
2026-04-13 12:12:52.974 | INFO     | src.model:train:385 - Epoch 2: Train Loss = 2.033226, Val Loss = 14.389959
2026-04-13 12:12:52.976 | INFO     | src.model:train:385 - Epoch 4: Train Loss = 2.033226, Val Loss = 14.389959
2026-04-13 12

In [10]:
with open("models/experiment_12.pkl", "wb") as f:
    pickle.dump(results_h1, f)

print([r["best_iteration"] for r in results_h1])
print([r["best_neurons"] for r in results_h1])

[9, 9, 9, 9, 9]
[45, 49, 49, 47, 44]


## Table 1: Effect of Gradient Data (PDPA v3)

Compare L2 loss (value only) vs H1 loss (value + gradient).

In [ ]:
import pickle
import pandas as pd
import numpy as np

with open("models/experiment_11.pkl", "rb") as f:
    results_l2_loaded = pickle.load(f)
with open("models/experiment_12.pkl", "rb") as f:
    results_h1_loaded = pickle.load(f)

def extract_table1_row(result):
    bi = result['best_iteration']
    return {
        'Err_L2': result['err_l2_train'][bi],
        'Err_H1': result['err_h1_train'][bi],
        'N': result['best_neurons'],
        'Err_L2_val': result['err_l2_val'][bi],
        'Err_H1_val': result['err_h1_val'][bi],
    }

# Use gamma=0 results (index 0)
row_l2 = extract_table1_row(results_l2_loaded[0])
row_h1 = extract_table1_row(results_h1_loaded[0])

# Training Errors
train_df = pd.DataFrame(
    [[f"{row_l2['Err_L2']:.2e}", f"{row_l2['Err_H1']:.2e}", row_l2['N']],
     [f"{row_h1['Err_L2']:.2e}", f"{row_h1['Err_H1']:.2e}", row_h1['N']]],
    index=["L2 loss (no gradient)", "H1 loss (with gradient)"],
    columns=["Err_L2", "Err_H1", "N (neurons)"]
)
print("Training Errors")
display(train_df)

# Validation Errors
val_df = pd.DataFrame(
    [[f"{row_l2['Err_L2_val']:.2e}", f"{row_l2['Err_H1_val']:.2e}", row_l2['N']],
     [f"{row_h1['Err_L2_val']:.2e}", f"{row_h1['Err_H1_val']:.2e}", row_h1['N']]],
    index=["L2 loss (no gradient)", "H1 loss (with gradient)"],
    columns=["Err_L2", "Err_H1", "N (neurons)"]
)
print("\nValidation Errors")
display(val_df)

## Table 2: Effect of Alpha (PDPA v3, gamma=0, L2 loss)

Sweep alpha in [1e-2, 1e-3, 1e-4, 1e-5] with gamma=0.

In [ ]:
from src.PDPA_v3 import PDPA_v3

alphas = [1e-2, 1e-3, 1e-4, 1e-5]
gamma_fixed = 0

results_alpha_v3 = []
for a in alphas:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    pdpa = PDPA_v3(
        data=data_dict, alpha=a, gamma=gamma_fixed,
        power=power, activation=activation,
        use_sphere=use_sphere, loss_weights="l2", verbose=False,
    )
    result = pdpa.retrain(
        num_iterations=num_iterations, num_insertion=num_insertions,
        threshold=pruning_threshold, verbose=False,
    )
    results_alpha_v3.append(result)
    bi = result['best_iteration']
    print(f"alpha={a:.0e}: N={result['best_neurons']}, err_l2={result['err_l2_train'][bi]:.2e}")

with open("models/experiment_14.pkl", "wb") as f:
    pickle.dump(results_alpha_v3, f)

In [ ]:
rows = []
for r in results_alpha_v3:
    bi = r['best_iteration']
    rows.append({
        'alpha': f"{r['alpha']:.0e}",
        'N': r['best_neurons'],
        '||N-f||': f"{r['err_l2_train'][bi]:.2e}",
    })

alpha_df = pd.DataFrame(rows).set_index('alpha')
display(alpha_df)

---

## Power = 2 (q = 2/3)

Repeat the experiments with `power=2.0` (`q = 2/(p+1) = 2/3`).

In [ ]:
power_2 = 2.0

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

#### L2 loss (power=2)

In [ ]:
from src.PDPA_v3 import PDPA_v3

results_l2_p2 = []

for gamma in gammas:
    pdpa = PDPA_v3(
        data=data_dict,
        alpha=alpha,
        gamma=gamma,
        power=power_2,
        activation=activation,
        use_sphere=use_sphere,
        loss_weights="l2",
        verbose=True,
    )

    result = pdpa.retrain(
        num_iterations=num_iterations,
        num_insertion=num_insertions,
        threshold=pruning_threshold,
        verbose=True,
    )
    results_l2_p2.append(result)

In [ ]:
with open("models/experiment_15.pkl", "wb") as f:
    pickle.dump(results_l2_p2, f)

print([r["best_iteration"] for r in results_l2_p2])
print([r["final_neurons"] for r in results_l2_p2])

#### H1 loss (power=2)

In [ ]:
from src.PDPA_v3 import PDPA_v3

results_h1_p2 = []

for gamma in gammas:
    pdpa = PDPA_v3(
        data=data_dict,
        alpha=alpha,
        gamma=gamma,
        power=power_2,
        activation=activation,
        use_sphere=use_sphere,
        loss_weights="h1",
        verbose=True,
    )

    result = pdpa.retrain(
        num_iterations=num_iterations,
        num_insertion=num_insertions,
        threshold=pruning_threshold,
        decorrelation=False,
        verbose=True,
    )
    results_h1_p2.append(result)

In [ ]:
with open("models/experiment_16.pkl", "wb") as f:
    pickle.dump(results_h1_p2, f)

print([r["best_iteration"] for r in results_h1_p2])
print([r["best_neurons"] for r in results_h1_p2])

### Table 1: Effect of Gradient Data (power=2)

In [ ]:
def extract_table1_row(result):
    bi = result['best_iteration']
    return {
        'Err_L2': result['err_l2_train'][bi],
        'Err_H1': result['err_h1_train'][bi],
        'N': result['best_neurons'],
        'Err_L2_val': result['err_l2_val'][bi],
        'Err_H1_val': result['err_h1_val'][bi],
    }

# Use gamma=0 results (index 0)
row_l2 = extract_table1_row(results_l2_p2[0])
row_h1 = extract_table1_row(results_h1_p2[0])

# Training Errors
train_df = pd.DataFrame(
    [[f"{row_l2['Err_L2']:.2e}", f"{row_l2['Err_H1']:.2e}", row_l2['N']],
     [f"{row_h1['Err_L2']:.2e}", f"{row_h1['Err_H1']:.2e}", row_h1['N']]],
    index=["L2 loss (no gradient)", "H1 loss (with gradient)"],
    columns=["Err_L2", "Err_H1", "N (neurons)"]
)
print("Training Errors (power=2)")
display(train_df)

# Validation Errors
val_df = pd.DataFrame(
    [[f"{row_l2['Err_L2_val']:.2e}", f"{row_l2['Err_H1_val']:.2e}", row_l2['N']],
     [f"{row_h1['Err_L2_val']:.2e}", f"{row_h1['Err_H1_val']:.2e}", row_h1['N']]],
    index=["L2 loss (no gradient)", "H1 loss (with gradient)"],
    columns=["Err_L2", "Err_H1", "N (neurons)"]
)
print("\nValidation Errors (power=2)")
display(val_df)

### Table 2: Effect of Alpha (power=2, gamma=0, L2 loss)

In [ ]:
from src.PDPA_v3 import PDPA_v3

alphas = [1e-2, 1e-3, 1e-4, 1e-5]
gamma_fixed = 0

results_alpha_v3_p2 = []
for a in alphas:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    pdpa = PDPA_v3(
        data=data_dict, alpha=a, gamma=gamma_fixed,
        power=power_2, activation=activation,
        use_sphere=use_sphere, loss_weights="l2", verbose=False,
    )
    result = pdpa.retrain(
        num_iterations=num_iterations, num_insertion=num_insertions,
        threshold=pruning_threshold, verbose=False,
    )
    results_alpha_v3_p2.append(result)
    bi = result['best_iteration']
    print(f"alpha={a:.0e}: N={result['best_neurons']}, err_l2={result['err_l2_train'][bi]:.2e}")

with open("models/experiment_17.pkl", "wb") as f:
    pickle.dump(results_alpha_v3_p2, f)

In [ ]:
rows = []
for r in results_alpha_v3_p2:
    bi = r['best_iteration']
    rows.append({
        'alpha': f"{r['alpha']:.0e}",
        'N': r['best_neurons'],
        '||N-f||': f"{r['err_l2_train'][bi]:.2e}",
    })

alpha_df_p2 = pd.DataFrame(rows).set_index('alpha')
display(alpha_df_p2)